<a href="https://colab.research.google.com/github/unaizanouman/Colab-Codes/blob/main/zePOP_LeaderElection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import heapq
import threading
import time
import random
from collections import defaultdict, deque
from typing import Dict, List, Set, Tuple, Optional
import math

class ZePoPNode:
    def __init__(self, node_id: int, network):
        self.id = node_id
        self.network = network
        self.neighbors = set()
        self.link_delays = {}

        # Phase 1 variables
        self.D_sx = {}
        self.phi_s = {}
        self.psi_sy = {}
        self.eta_sy = {}
        self.O_xy = defaultdict(int)
        self.I_xy = defaultdict(int)

        # Branch weight information
        self.T_x_C = defaultdict(float)
        self.T_y_C = defaultdict(float)

        # Phase 2 variables
        self.is_candidate = False
        self.closeness_centrality = 0.0
        self.current_leader = None
        self.leader_direction = None

        # DCDT variables
        self.parent = None
        self.children = set()
        self.fg_list = set()

        # Message handling
        self.message_queue = deque()
        self.lock = threading.Lock()
        self.processed_sources = set()

        # Initialize with self
        self.D_sx[self.id] = 0.0
        self.phi_s[self.id] = True

    def add_neighbor(self, neighbor_id: int, delay: float):
        self.neighbors.add(neighbor_id)
        self.link_delays[neighbor_id] = delay
        self.O_xy[neighbor_id] = 0
        self.I_xy[neighbor_id] = 0
        self.T_x_C[neighbor_id] = 0.0
        self.T_y_C[neighbor_id] = 0.0

    def receive_message(self, message: Dict):
        with self.lock:
            self.message_queue.append(message)

    def process_messages(self):
        processed_count = 0
        while self.message_queue:
            message = self.message_queue.popleft()
            self.handle_message(message)
            processed_count += 1
        return processed_count

    def handle_message(self, message: Dict):
        msg_type = message.get('type')

        if msg_type == 'ELECTION':
            self.handle_election_message(message)
        elif msg_type == 'INFORM':
            self.handle_inform_message(message)
        elif msg_type == 'JOIN':
            self.handle_join_message(message)
        elif msg_type == 'LEAVE':
            self.handle_leave_message(message)

    def handle_election_message(self, message: Dict):
        source_id = message['source_id']
        via_neighbor = message['via_neighbor']
        cumulative_delay = message['cumulative_delay']
        link_delay = message['link_delay']

        # Store that we've heard from this source
        self.processed_sources.add(source_id)

        # Calculate D'_sx
        actual_link_delay = self.link_delays.get(via_neighbor, float('inf'))
        D_prime_sx = cumulative_delay + (link_delay + actual_link_delay) / 2

        is_first_message = source_id not in self.phi_s or not self.phi_s[source_id]
        current_best_delay = self.D_sx.get(source_id, float('inf'))

        if is_first_message:
            # First message from this source
            self.phi_s[source_id] = True
            self.D_sx[source_id] = D_prime_sx
            self.accept_election_message(source_id, via_neighbor, cumulative_delay)
            self.forward_election_message(source_id, via_neighbor)
            print(f"Node {self.id}: First election message from {source_id} via {via_neighbor}, delay: {D_prime_sx:.2f}")

        elif D_prime_sx < current_best_delay:
            # Better path found
            print(f"Node {self.id}: Better path from {source_id} via {via_neighbor}, new delay: {D_prime_sx:.2f} (was {current_best_delay:.2f})")
            self.reset_direction(source_id)
            self.D_sx[source_id] = D_prime_sx
            self.accept_election_message(source_id, via_neighbor, cumulative_delay)
            self.adjust_send(source_id, via_neighbor)
            self.forward_election_message(source_id, via_neighbor)

        elif D_prime_sx == current_best_delay:
            # Equally good path
            self.accept_election_message(source_id, via_neighbor, cumulative_delay)
            self.adjust_send(source_id, via_neighbor)

        else:
            # Check for set C classification
            if current_best_delay + actual_link_delay > cumulative_delay:
                # Node s is in set C
                self.T_x_C[via_neighbor] += current_best_delay
                self.T_y_C[via_neighbor] += cumulative_delay
                self.adjust_send(source_id, via_neighbor)

    def accept_election_message(self, source_id: int, via_neighbor: int, cumulative_delay: float):
        if not self.eta_sy.get((source_id, via_neighbor), False):
            self.eta_sy[(source_id, via_neighbor)] = True
            self.I_xy[via_neighbor] += 1

    def adjust_send(self, source_id: int, via_neighbor: int):
        if self.psi_sy.get((source_id, via_neighbor), False):
            self.psi_sy[(source_id, via_neighbor)] = False
            self.O_xy[via_neighbor] -= 1

    def forward_election_message(self, source_id: int, exclude_neighbor: int):
        for neighbor in self.neighbors:
            if neighbor == exclude_neighbor:
                continue

            neighbor_delay = self.link_delays[neighbor]
            path_delay_to_neighbor = self.D_sx[source_id] + neighbor_delay

            if not self.psi_sy.get((source_id, neighbor), False):
                self.psi_sy[(source_id, neighbor)] = True
                self.O_xy[neighbor] += 1

                message = {
                    'type': 'ELECTION',
                    'source_id': source_id,
                    'via_neighbor': self.id,
                    'cumulative_delay': self.D_sx[source_id],
                    'link_delay': neighbor_delay
                }
                self.network.send_message(self.id, neighbor, message)

    def reset_direction(self, source_id: int):
        for neighbor in list(self.neighbors):
            if self.eta_sy.get((source_id, neighbor), False):
                self.eta_sy[(source_id, neighbor)] = False
                self.I_xy[neighbor] = max(0, self.I_xy[neighbor] - 1)

    def handle_inform_message(self, message: Dict):
        candidate_id = message['candidate_id']
        centrality = message['centrality']
        via_neighbor = message['via_neighbor']

        if (self.current_leader is None or
            centrality > self.closeness_centrality or
            (centrality == self.closeness_centrality and candidate_id < self.current_leader)):

            self.current_leader = candidate_id
            self.closeness_centrality = centrality
            self.leader_direction = via_neighbor
            print(f"Node {self.id}: Updated leader to {candidate_id} with centrality {centrality:.3f}")

    def start_election_phase1(self, total_nodes: int):
        print(f"Node {self.id}: Starting election phase 1")
        self.processed_sources.clear()

        # Broadcast election message to all neighbors
        for neighbor in self.neighbors:
            message = {
                'type': 'ELECTION',
                'source_id': self.id,
                'via_neighbor': self.id,
                'cumulative_delay': 0.0,
                'link_delay': self.link_delays[neighbor]
            }
            self.network.send_message(self.id, neighbor, message)

    def start_election_phase2(self, total_nodes: int):
        print(f"Node {self.id}: Starting election phase 2")

        # Calculate total delay T_x (sum of all shortest path delays)
        T_x = sum(delay for s, delay in self.D_sx.items() if s != self.id)

        # Calculate closeness centrality
        if T_x > 0 and len(self.D_sx) > 1:
            self.closeness_centrality = (total_nodes - 1) / T_x
        else:
            self.closeness_centrality = 0.0

        # Check candidacy condition against all neighbors
        self.is_candidate = True
        for neighbor in self.neighbors:
            if not self.is_better_candidate_than(neighbor, total_nodes):
                self.is_candidate = False
                break

        print(f"Node {self.id}: Closeness centrality = {self.closeness_centrality:.3f}, Candidate = {self.is_candidate}")

        # If candidate, inform others
        if self.is_candidate:
            self.broadcast_inform_message()

    def is_better_candidate_than(self, neighbor: int, total_nodes: int) -> bool:
        d_xy = self.link_delays.get(neighbor, float('inf'))

        if d_xy > 0:
            O_xy = self.O_xy[neighbor]
            I_xy = self.I_xy[neighbor]
            T_x_C = self.T_x_C[neighbor]
            T_y_C = self.T_y_C[neighbor]

            delta_xy = O_xy - (I_xy + (T_x_C - T_y_C) / d_xy)
        else:
            T_x_C = self.T_x_C[neighbor]
            T_y_C = self.T_y_C[neighbor]
            delta_xy = T_y_C - T_x_C

        # Determine superiority
        if delta_xy > 0:
            return True
        elif delta_xy < 0:
            return False
        else:
            # Tie-breaking
            if self.current_leader == self.id:
                return True
            elif self.current_leader == neighbor:
                return False
            else:
                return self.id < neighbor

    def broadcast_inform_message(self):
        print(f"Node {self.id}: Broadcasting INFORM message as candidate")
        for neighbor in self.neighbors:
            message = {
                'type': 'INFORM',
                'candidate_id': self.id,
                'centrality': self.closeness_centrality,
                'via_neighbor': self.id
            }
            self.network.send_message(self.id, neighbor, message)

    def build_dcdt(self):
        if self.current_leader == self.id:
            self.parent = None
            print(f"Node {self.id}: I am the leader (root of DCDT)")
        else:
            if self.leader_direction and self.leader_direction in self.neighbors:
                self.parent = self.leader_direction
            else:
                min_delay = float('inf')
                for neighbor in self.neighbors:
                    estimated_delay = self.link_delays[neighbor]
                    if estimated_delay < min_delay:
                        min_delay = estimated_delay
                        self.parent = neighbor
            print(f"Node {self.id}: Parent in DCDT = {self.parent}")

    def handle_join_message(self, message: Dict):
        new_node_id = message['new_node_id']
        print(f"Node {self.id}: Handling JOIN from node {new_node_id}")

    def handle_leave_message(self, message: Dict):
        leaving_node_id = message['leaving_node_id']
        print(f"Node {self.id}: Handling LEAVE from node {leaving_node_id}")


class ZePoPNetwork:
    def __init__(self):
        self.nodes = {}
        self.current_leader = None
        self.total_nodes = 0
        self.message_queue = deque()
        self.message_history = []

    def add_node(self, node_id: int):
        if node_id not in self.nodes:
            self.nodes[node_id] = ZePoPNode(node_id, self)
            self.total_nodes = len(self.nodes)
            print(f"Added node {node_id}")

    def add_link(self, node1: int, node2: int, delay: float):
        if node1 in self.nodes and node2 in self.nodes:
            self.nodes[node1].add_neighbor(node2, delay)
            self.nodes[node2].add_neighbor(node1, delay)
            print(f"Added link {node1} <-> {node2} with delay {delay}")

    def send_message(self, from_node: int, to_node: int, message: Dict):
        if to_node in self.nodes:
            self.message_queue.append((from_node, to_node, message))
            self.message_history.append((from_node, to_node, message['type']))

    def process_messages(self, max_iterations=1000):
        print(f"Processing messages in queue: {len(self.message_queue)}")
        iteration = 0
        total_processed = 0

        while self.message_queue and iteration < max_iterations:
            from_node, to_node, message = self.message_queue.popleft()

            if to_node in self.nodes:
                self.nodes[to_node].receive_message(message)
                processed = self.nodes[to_node].process_messages()
                total_processed += processed

            iteration += 1

        print(f"Processed {total_processed} messages in {iteration} iterations")
        return total_processed

    def start_election(self):
        print("\n" + "="*60)
        print("STARTING ZEPOP LEADER ELECTION PROTOCOL")
        print("="*60)

        # Reset all nodes
        for node in self.nodes.values():
            node.D_sx = {node.id: 0.0}
            node.phi_s = {node.id: True}
            node.psi_sy.clear()
            node.eta_sy.clear()
            node.O_xy.clear()
            node.I_xy.clear()
            node.T_x_C.clear()
            node.T_y_C.clear()
            node.is_candidate = False
            node.closeness_centrality = 0.0
            node.current_leader = None

        # Phase 1: Calculate shortest paths
        print("\n--- PHASE 1: Shortest Path Calculation ---")
        for node in self.nodes.values():
            node.start_election_phase1(self.total_nodes)

        # Process all election messages
        self.process_messages(2000)

        # Wait a bit for messages to propagate
        time.sleep(0.1)
        self.process_messages(1000)

        # Phase 2: Leader selection
        print("\n--- PHASE 2: Leader Election ---")
        for node in self.nodes.values():
            node.start_election_phase2(self.total_nodes)

        # Process all inform messages
        self.process_messages(1000)

        # Determine final leader
        leader_id = None
        max_centrality = -1

        for node in self.nodes.values():
            if node.closeness_centrality > max_centrality:
                max_centrality = node.closeness_centrality
                leader_id = node.id
            elif (node.closeness_centrality == max_centrality and
                  leader_id is not None and node.id < leader_id):
                leader_id = node.id

        self.current_leader = leader_id

        # Build DCDT
        print("\n--- Building DCDT ---")
        for node in self.nodes.values():
            node.build_dcdt()

        print(f"\n🎉 ELECTION COMPLETED: Leader is Node {leader_id}")
        return leader_id

    def print_election_results(self):
        print("\n" + "="*60)
        print("ELECTION RESULTS")
        print("="*60)

        leader_id = self.current_leader
        for node_id in sorted(self.nodes.keys()):
            node = self.nodes[node_id]
            status = " 🏆 LEADER" if node_id == leader_id else " 💡 Candidate" if node.is_candidate else ""
            print(f"Node {node_id}:")
            print(f"  Closeness Centrality: {node.closeness_centrality:.4f}{status}")
            print(f"  Parent in DCDT: {node.parent}")
            print(f"  Neighbors: {sorted(node.neighbors)}")
            print(f"  Shortest paths to {len(node.D_sx)} nodes")

            # Print some path delays
            if node.D_sx:
                sample_targets = list(node.D_sx.keys())[:3]
                delays_str = ", ".join([f"→{t}:{d:.1f}" for t, d in list(node.D_sx.items())[:3]])
                print(f"  Sample delays: [{delays_str}]")
            print()


def create_example_network():
    """Create the example network from the paper (Fig. 6)"""
    network = ZePoPNetwork()

    # Add nodes 0-4 as in the paper example
    for i in range(5):
        network.add_node(i)

    # Add links with delays (blue numbers from Fig. 6)
    links = [
        (0, 1, 2), (0, 2, 3), (0, 3, 4),
        (1, 2, 1), (1, 3, 2), (1, 4, 3),
        (2, 3, 2), (3, 4, 1)
    ]

    for node1, node2, delay in links:
        network.add_link(node1, node2, delay)

    return network


def test_zepp_protocol():
    """Test the ZePoP protocol with the example network"""
    print("Testing ZePoP Leader Election Protocol")
    print("Creating example network from paper (Fig. 6)...")

    # Create example network
    network = create_example_network()

    # Run election
    leader_id = network.start_election()

    # Print detailed results
    network.print_election_results()

    # Verify optimal leader (should be node 1 as in paper example)
    expected_leader = 1
    if leader_id == expected_leader:
        print(f"✅ SUCCESS: Correct leader elected (Node {expected_leader})")
        print(f"   Expected centrality: ~0.714")
        actual_centrality = network.nodes[expected_leader].closeness_centrality
        print(f"   Actual centrality: {actual_centrality:.3f}")
    else:
        print(f"❌ FAILURE: Expected Node {expected_leader}, but got Node {leader_id}")

    return network


# Simple test without complex dependencies
def quick_test():
    """A simpler test that's more likely to work in Colab"""
    print("🔍 Running Quick ZePoP Test...")

    # Create a simple 3-node network
    network = ZePoPNetwork()

    # Add 3 nodes in a triangle
    for i in [0, 1, 2]:
        network.add_node(i)

    # Add links with reasonable delays
    network.add_link(0, 1, 2.0)
    network.add_link(1, 2, 1.5)
    network.add_link(2, 0, 3.0)

    # Run election
    leader = network.start_election()

    # Show results
    print(f"\nQuick Test Results:")
    print(f"Network has {network.total_nodes} nodes")
    print(f"Elected leader: Node {leader}")

    for node_id, node in network.nodes.items():
        status = " (LEADER)" if node_id == leader else ""
        print(f"Node {node_id}: centrality = {node.closeness_centrality:.3f}{status}")


if __name__ == "__main__":
    print("ZePoP Protocol Implementation")
    print("=" * 50)

    # Run the quick test first
    quick_test()

    print("\n" + "="*50)
    print("Now running full example from paper...")
    print("="*50)

    # Then run the full example
    test_network = test_zepp_protocol()

ZePoP Protocol Implementation
🔍 Running Quick ZePoP Test...
Added node 0
Added node 1
Added node 2
Added link 0 <-> 1 with delay 2.0
Added link 1 <-> 2 with delay 1.5
Added link 2 <-> 0 with delay 3.0

STARTING ZEPOP LEADER ELECTION PROTOCOL

--- PHASE 1: Shortest Path Calculation ---
Node 0: Starting election phase 1
Node 1: Starting election phase 1
Node 2: Starting election phase 1
Processing messages in queue: 6
Node 1: First election message from 0 via 0, delay: 2.00
Node 2: First election message from 0 via 0, delay: 3.00
Node 0: First election message from 1 via 1, delay: 2.00
Node 2: First election message from 1 via 1, delay: 1.50
Node 0: First election message from 2 via 2, delay: 3.00
Node 1: First election message from 2 via 2, delay: 1.50
Processed 12 messages in 12 iterations
Processing messages in queue: 0
Processed 0 messages in 0 iterations

--- PHASE 2: Leader Election ---
Node 0: Starting election phase 2
Node 0: Closeness centrality = 0.400, Candidate = False
Node 1